In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from optional_fine_tune import BertForTweetClassification, DfToDataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
test = pd.read_csv("data/test.csv")

test['keyword'] = test['keyword'].fillna("")
test['combined_text'] = "Keyword: " + test['keyword'] + ". Text: " + test['text']

test_texts = test['combined_text']
test_targets = np.zeros(len(test_texts))

In [ ]:
test_dataset = DfToDataset(test_texts, test_targets, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
print("Загрузка весов моделей с диска...")
models = []

for fold in range(5):
    model = BertForTweetClassification(MODEL_NAME)
    model_path = f"models/best_bert_fold_{fold + 1}.pt"

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Файл весов {model_path} не найден! Убедись, что он лежит в папке со скриптом.")

    weights = torch.load(model_path, map_location=device)
    model.load_state_dict(weights)
    model.to(device)
    model.eval()  # Отключаем Dropout!
    models.append(model)
print("Все 5 моделей успешно загружены в память.")

In [ ]:
print("Запуск предсказания на тестовых данных...")
all_blend_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        batch_probs = []

        for model in models:
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(logits, dim=1)
            batch_probs.append(probs[:, 1].cpu().numpy())

        avg_probs = np.mean(batch_probs, axis=0)
        all_blend_preds.extend(avg_probs)

final_labels = [1 if prob >= 0.5 else 0 for prob in all_blend_preds]

submission = pd.DataFrame({
    "id": test["id"],
    "target": final_labels
})

submission.to_csv("results/submission_ensemble.csv", index=False)